In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ── pipeline_config: tells pipeline WHAT to run ──
config_data = [
    ("silver", "nb_bronze_to_silver_orders",    1, True),
    ("silver", "nb_bronze_to_silver_customers", 1, True),
    ("silver", "nb_bronze_to_silver_products",  1, True),
    ("gold",   "nb_silver_to_gold",             2, False),
]

config_schema = T.StructType([
    T.StructField("stage",          T.StringType(),  False),
    T.StructField("notebook_name",  T.StringType(),  False),
    T.StructField("exec_order",     T.IntegerType(), False),
    T.StructField("is_parallel",    T.BooleanType(), False),
])

spark.createDataFrame(config_data, config_schema) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("pipeline_config")

# ── pipeline_log: records every run ──
from pyspark.sql import types as T

log_schema = T.StructType([
    T.StructField("run_id",           T.StringType(),    True),
    T.StructField("pipeline_name",    T.StringType(),    True),
    T.StructField("trigger_type",     T.StringType(),    True),
    T.StructField("stage",            T.StringType(),    True),
    T.StructField("notebook_name",    T.StringType(),    True),
    T.StructField("status",           T.StringType(),    True),
    T.StructField("start_time",       T.TimestampType(), True),
    T.StructField("end_time",         T.TimestampType(), True),
    T.StructField("duration_seconds", T.LongType(),      True),
    T.StructField("rows_processed",   T.LongType(),      True),
    T.StructField("error_message",    T.StringType(),    True),
    T.StructField("load_date",        T.StringType(),    True),
])

spark.createDataFrame([], log_schema) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("pipeline_log")

# ── watermark_control: tracks last run per source ──
watermark_data = [
    ("orders",    "1900-01-01 00:00:00"),
    ("customers", "1900-01-01 00:00:00"),
    ("products",  "1900-01-01 00:00:00"),
]

watermark_schema = T.StructType([
    T.StructField("source_name",       T.StringType(), False),
    T.StructField("last_loaded_value", T.StringType(), False),
])

spark.createDataFrame(watermark_data, watermark_schema) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("watermark_control")

print("Control framework ready:")
print("  pipeline_config   ✅")
print("  pipeline_log      ✅")
print("  watermark_control ✅")


StatementMeta(, bdb5a329-79db-4622-ae0f-186d5838eaff, 4, Finished, Available, Finished, False)

Control framework ready:
  pipeline_config   ✅
  pipeline_log      ✅
  watermark_control ✅
